# Model Integration Demo Notebook

This notebook loads the saved Logistic Regression, Random Forest, and XGBoost model artifacts, applies the correct preprocessing for each one, and compares their churn predictions on selected customer records.


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from IPython.display import display

# Update these paths if your files live somewhere else
DATA_PATH = 'kkbox_dataset.csv'
LOG_MODEL_PATH = 'logistic_model.pkl'
LOG_PREPROCESS_PATH = 'logistic_preprocess.pkl'
RF_MODEL_PATH = 'random_forest_model.pkl'
RF_PREPROCESS_PATH = 'random_forest_preprocess.pkl'
XGB_MODEL_PATH = 'xgboost_model.pkl'
XGB_PREPROCESS_PATH = 'xgboost_preprocess.pkl'

print('Working directory:', os.getcwd())

In [ ]:
def require_file(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing file: {path}')
    return path

for path in [
    DATA_PATH,
    LOG_MODEL_PATH, LOG_PREPROCESS_PATH,
    RF_MODEL_PATH, RF_PREPROCESS_PATH,
    XGB_MODEL_PATH, XGB_PREPROCESS_PATH,
]:
    require_file(path)

print('All required files were found.')

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
display(df.head())

log_model = joblib.load(LOG_MODEL_PATH)
log_preprocess = joblib.load(LOG_PREPROCESS_PATH)

rf_model = joblib.load(RF_MODEL_PATH)
rf_preprocess = joblib.load(RF_PREPROCESS_PATH)

xgb_model = joblib.load(XGB_MODEL_PATH)
xgb_preprocess = joblib.load(XGB_PREPROCESS_PATH)

print('Models and preprocessing artifacts loaded successfully.')

In [ ]:
def preprocess_for_logistic(row_df, preprocess):
    row = row_df.copy()
    row = row.drop(columns=['is_churn', 'msno'], errors='ignore')

    numeric_cols = preprocess['numeric_cols']
    categorical_cols = preprocess['categorical_cols']
    numeric_medians = preprocess['numeric_medians']
    feature_columns = preprocess['feature_columns']

    for col in numeric_cols:
        if col in row.columns:
            row[col] = pd.to_numeric(row[col], errors='coerce')
            row[col] = row[col].fillna(numeric_medians.get(col, 0.0))

    for col in categorical_cols:
        if col in row.columns:
            row[col] = row[col].astype(str).fillna('Unknown')
        else:
            row[col] = 'Unknown'

    row = pd.get_dummies(row, drop_first=True)
    row = row.reindex(columns=feature_columns, fill_value=0)
    return row.astype('float32')


def preprocess_for_tree_model(row_df, preprocess):
    row = row_df.copy()
    row = row.drop(columns=['is_churn', 'msno'], errors='ignore')

    numeric_cols = preprocess['numeric_cols']
    categorical_cols = preprocess['categorical_cols']
    numeric_medians = preprocess['numeric_medians']
    encoder_maps = preprocess['encoder_maps']
    feature_columns = preprocess['feature_columns']

    for col in numeric_cols:
        if col in row.columns:
            row[col] = pd.to_numeric(row[col], errors='coerce')
            row[col] = row[col].fillna(numeric_medians.get(col, 0.0))
        else:
            row[col] = numeric_medians.get(col, 0.0)

    for col in categorical_cols:
        if col in row.columns:
            row[col] = row[col].astype(str).fillna('Unknown')
        else:
            row[col] = 'Unknown'

        mapping = encoder_maps.get(col, {})
        row[col] = row[col].map(lambda x: mapping.get(x, mapping.get('Unknown', -1)))

    row = row.reindex(columns=feature_columns, fill_value=0)
    return row


In [ ]:
def compare_models_for_index(index):
    if index not in df.index:
        raise IndexError(f'Index {index} not found in dataset.')

    row_df = df.loc[[index]].copy()
    actual = row_df['is_churn'].iloc[0] if 'is_churn' in row_df.columns else None

    X_log = preprocess_for_logistic(row_df, log_preprocess)
    X_rf = preprocess_for_tree_model(row_df, rf_preprocess)
    X_xgb = preprocess_for_tree_model(row_df, xgb_preprocess)

    log_pred = int(log_model.predict(X_log)[0])
    log_prob = float(log_model.predict_proba(X_log)[0][1])

    rf_pred = int(rf_model.predict(X_rf)[0])
    rf_prob = float(rf_model.predict_proba(X_rf)[0][1])

    xgb_pred = int(xgb_model.predict(X_xgb)[0])
    xgb_prob = float(xgb_model.predict_proba(X_xgb)[0][1])

    results = pd.DataFrame({
        'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
        'Prediction': [log_pred, rf_pred, xgb_pred],
        'Churn Probability': [log_prob, rf_prob, xgb_prob],
        'Actual is_churn': [actual, actual, actual]
    })

    return row_df, results


In [ ]:
# Change this index to test different customer rows
selected_index = int(df.index[0])

selected_row, comparison = compare_models_for_index(selected_index)
print(f'Selected index: {selected_index}')
display(selected_row.head())
display(comparison)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.bar(comparison['Model'], comparison['Churn Probability'])
plt.ylim(0, 1)
plt.ylabel('Churn Probability')
plt.title('Model Comparison for Selected Customer')
plt.show()

In [ ]:
# Optional widget version for easier interaction in Jupyter
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    index_widget = widgets.IntSlider(
        value=int(df.index[0]),
        min=int(df.index.min()),
        max=int(df.index.max()),
        step=1,
        description='Row index:'
    )

    output = widgets.Output()

    def on_change(change):
        if change['name'] == 'value':
            with output:
                clear_output(wait=True)
                _, results = compare_models_for_index(change['new'])
                display(results)

    index_widget.observe(on_change)
    display(index_widget)
    display(output)

    with output:
        _, results = compare_models_for_index(index_widget.value)
        display(results)

except Exception as e:
    print('ipywidgets not available or failed to load:', e)
    print('You can still use the selected_index cell above.')